# Entropy regularization report

Target study: `resnet50_aig_plus_neg_entropy_coef_lambda1em4_120ep_repeats3_ordered`.

This is the 12-run entropy contribution check: 4 values of `model.entropy_regularization_coef` (`0.0`, `0.1`, `0.3`, `1.0`) times 3 repeats. The notebook keeps the old core plots (accuracy, open/closed blocks or channels, lambda, gradient norms) and adds beta-level diagnostics for whether the entropy term changed optimization or compute/accuracy tradeoffs.

Useful new plots:

- accuracy by epoch, averaged by beta with repeat spread;
- active FLOPs / active blocks / mean gate probability by beta;
- posterior entropy and the signed entropy loss term by beta;
- best/final accuracy and compute as beta boxplots;
- accuracy-vs-compute scatter colored by beta;
- `grad(regularization) / grad(CE)` ratios for the whole model and gate logits.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import plotly.graph_objects as go

REPO_ROOT = Path.cwd().resolve()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / "src" / "net_complexity").exists():
    REPO_ROOT = REPO_ROOT.parent

if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))

from net_complexity.studies import (
    gradient_norm_catalog,
    load_study,
    plot_core_statistics,
    plot_gradient_norms,
    plot_metric,
    print_available_metrics,
)

In [ ]:
STUDY_NAME = "resnet50_aig_plus_neg_entropy_coef_lambda1em4_120ep_repeats3_ordered"
EXPECTED_BETAS = (0.0, 0.1, 0.3, 1.0)
EXPECTED_REPEATS_PER_BETA = 3
EXPECTED_RUNS = len(EXPECTED_BETAS) * EXPECTED_REPEATS_PER_BETA
STRICT_EXPECTED_SHAPE = True

# Set this if the run directory is not under REPO_ROOT / "outputs/studies".
STUDY_DIR_OVERRIDE = None


def _find_latest_study_dir() -> Path:
    if STUDY_DIR_OVERRIDE is not None:
        return Path(STUDY_DIR_OVERRIDE).expanduser().resolve()

    studies_root = REPO_ROOT / "outputs" / "studies"
    candidates = sorted(
        p for p in studies_root.glob(f"*{STUDY_NAME}")
        if p.is_dir() and (p / "runs").is_dir()
    )
    if not candidates:
        raise FileNotFoundError(
            f"No local study matching '*{STUDY_NAME}' was found under {studies_root}. "
            "Set STUDY_DIR_OVERRIDE to the exact study directory."
        )
    return candidates[-1]


STUDY_DIR = _find_latest_study_dir()
print("REPO_ROOT:", REPO_ROOT)
print("STUDY_DIR:", STUDY_DIR)

In [ ]:
summary_df, history_df = load_study(STUDY_DIR)


def _find_col(df: pd.DataFrame, candidates: list[str] | tuple[str, ...]) -> str | None:
    return next((col for col in candidates if col in df.columns), None)


def _numeric_col(df: pd.DataFrame, candidates: list[str] | tuple[str, ...], default=np.nan) -> pd.Series:
    col = _find_col(df, candidates)
    if col is None:
        return pd.Series(default, index=df.index, dtype="float64")
    return pd.to_numeric(df[col], errors="coerce")


def _string_col(
    df: pd.DataFrame,
    candidates: list[str] | tuple[str, ...],
    default: str = "unknown",
) -> pd.Series:
    col = _find_col(df, candidates)
    if col is None:
        return pd.Series(default, index=df.index, dtype="object")
    return df[col].astype("string").fillna(default).astype("object")


def _add_entropy_columns(
    summary_df: pd.DataFrame,
    history_df: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    summary_df = summary_df.copy()
    history_df = history_df.copy()

    summary_df["entropy_beta"] = _numeric_col(
        summary_df,
        (
            "label_beta",
            "model.entropy_regularization_coef",
            "mlflow.tags.entropy_regularization_coef",
        ),
    )
    summary_df["entropy_mode"] = _string_col(
        summary_df,
        ("model.entropy_regularization", "mlflow.tags.entropy_regularization"),
    )
    summary_df = summary_df.sort_values(["entropy_beta", "run_name"]).reset_index(drop=True)
    summary_df["repeat_idx"] = summary_df.groupby("entropy_beta", dropna=False).cumcount() + 1

    run_meta = summary_df[["run_name", "entropy_beta", "entropy_mode", "repeat_idx"]].drop_duplicates("run_name")
    run_meta = run_meta.set_index("run_name")
    for col in run_meta.columns:
        history_df[col] = history_df["run_name"].map(run_meta[col])

    if "lambda_coef" in history_df.columns:
        lambda_values = pd.to_numeric(history_df["lambda_coef"], errors="coerce")
    else:
        lambda_values = pd.Series(np.nan, index=history_df.index, dtype="float64")

    for split in ("train", "valid"):
        neg_entropy_col = f"{split}_negative_entropy"
        mean_p_open_col = f"{split}_mean_p_open"
        if neg_entropy_col in history_df.columns:
            negative_entropy = pd.to_numeric(history_df[neg_entropy_col], errors="coerce")
            history_df[f"{split}_entropy"] = -negative_entropy
            history_df[f"{split}_entropy_loss_term"] = history_df["entropy_beta"] * negative_entropy
        if mean_p_open_col in history_df.columns:
            mean_p_open = pd.to_numeric(history_df[mean_p_open_col], errors="coerce")
            history_df[f"{split}_lambda_p_open_loss"] = lambda_values * mean_p_open
        if (
            f"{split}_entropy_loss_term" in history_df.columns
            and f"{split}_lambda_p_open_loss" in history_df.columns
        ):
            history_df[f"{split}_estimated_gate_loss"] = (
                history_df[f"{split}_lambda_p_open_loss"]
                + history_df[f"{split}_entropy_loss_term"]
            )

    return summary_df, history_df


summary_df, history_df = _add_entropy_columns(summary_df, history_df)

ACC_COL = _find_col(
    history_df,
    ("valid_accuracy", "val_accuracy", "valid_acc", "val_acc", "accuracy"),
)
if ACC_COL is None:
    raise ValueError("No validation accuracy column found in history_df")

print(f"runs: {history_df['run_name'].nunique()}, history: {history_df.shape}")
print(f"accuracy column: {ACC_COL}")
display(summary_df)

In [ ]:
beta_counts = (
    summary_df.groupby("entropy_beta", dropna=False)["run_name"]
    .nunique()
    .rename("runs")
    .reset_index()
)
display(beta_counts)

runs_loaded = int(history_df["run_name"].nunique())
observed_betas = tuple(float(x) for x in sorted(summary_df["entropy_beta"].dropna().unique()))

if STRICT_EXPECTED_SHAPE:
    if runs_loaded != EXPECTED_RUNS:
        raise AssertionError(f"Expected {EXPECTED_RUNS} runs, got {runs_loaded}")
    if observed_betas != EXPECTED_BETAS:
        raise AssertionError(f"Expected betas {EXPECTED_BETAS}, got {observed_betas}")
    bad_counts = beta_counts[beta_counts["runs"] != EXPECTED_REPEATS_PER_BETA]
    if not bad_counts.empty:
        raise AssertionError(f"Expected {EXPECTED_REPEATS_PER_BETA} repeats per beta, got:\n{bad_counts}")

print("Study shape check passed.")

In [ ]:
ENDPOINT_METRICS = [
    "lambda_coef",
    "valid_loss",
    "valid_ce_loss",
    "valid_reg_loss",
    "valid_regularization_loss",
    "valid_mean_p_open",
    "valid_negative_entropy",
    "valid_entropy",
    "valid_lambda_p_open_loss",
    "valid_entropy_loss_term",
    "valid_estimated_gate_loss",
    "valid_active_blocks_expected",
    "valid_inactive_blocks_expected",
    "valid_mean_gate_prob",
    "valid_aig_static_flops_per_sample",
    "valid_aig_active_flops_per_sample",
    "valid_aig_skipped_flops_per_sample",
    "valid_aig_flops_skip_ratio",
    "valid_aig_flops_active_ratio",
    "grad_norm_ce_total_mean",
    "grad_norm_regularization_total_mean",
    "grad_norm_total_total_mean",
    "grad_norm_ce_gumbel_logits_total_mean",
    "grad_norm_regularization_gumbel_logits_total_mean",
    "grad_norm_total_gumbel_logits_total_mean",
]


def make_entropy_run_summary(history_df: pd.DataFrame, acc_col: str = ACC_COL) -> pd.DataFrame:
    rows = []
    for run_name, group in history_df.groupby("run_name", sort=False):
        group = group.sort_values("epoch") if "epoch" in group.columns else group.copy()
        acc_values = pd.to_numeric(group[acc_col], errors="coerce")
        best_row = group.loc[acc_values.idxmax()] if acc_values.notna().any() else group.iloc[-1]
        final_row = group.iloc[-1]
        row = {
            "run_name": run_name,
            "run_label": final_row.get("run_label", run_name),
            "entropy_beta": final_row.get("entropy_beta"),
            "entropy_mode": final_row.get("entropy_mode"),
            "repeat_idx": final_row.get("repeat_idx"),
            "best_epoch": best_row.get("epoch"),
            "final_epoch": final_row.get("epoch"),
            "best_valid_accuracy": best_row.get(acc_col),
            "final_valid_accuracy": final_row.get(acc_col),
            "accuracy_gap_final_minus_best": final_row.get(acc_col) - best_row.get(acc_col),
            "run_dir": final_row.get("run_dir"),
        }
        for metric in ENDPOINT_METRICS:
            if metric in group.columns:
                row[f"best_{metric}"] = best_row.get(metric)
                row[f"final_{metric}"] = final_row.get(metric)
        rows.append(row)
    return pd.DataFrame(rows).sort_values(["entropy_beta", "repeat_idx", "run_name"]).reset_index(drop=True)


run_summary_df = make_entropy_run_summary(history_df)
display(run_summary_df)

## Old core plots

These are the same first-pass checks used in the previous study notebooks: accuracy, factual open/closed counts, lambda, and the compact total gradient norm view.

In [ ]:
try:
    old_figures = plot_core_statistics(history_df, interactive=True)
except ValueError as exc:
    print(f"plot_core_statistics skipped: {exc}")

catalog = gradient_norm_catalog(history_df)
display(catalog)

if catalog.empty:
    print("No grad_norm_* columns found.")
else:
    for parameter_group in ("total", "gumbel_logits_total"):
        if parameter_group not in set(catalog["parameter_group"]):
            continue
        try:
            plot_gradient_norms(
                history_df,
                parameter_group=parameter_group,
                statistic="mean",
                yscale="log",
                interactive=True,
            )
        except ValueError as exc:
            print(f"plot_gradient_norms skipped for {parameter_group}: {exc}")

## Entropy contribution plots

The key comparison is beta `0.0` against `0.1`, `0.3`, and `1.0`. A useful entropy contribution should show a coherent shift in entropy/gate behavior or gradient balance, and ideally improve the accuracy-vs-compute tradeoff rather than only changing final accuracy within repeat noise.

In [ ]:
COLORS = ("#2563eb", "#dc2626", "#16a34a", "#9333ea", "#ea580c", "#0891b2")


def _hex_to_rgba(color: str, alpha: float = 0.16) -> str:
    color = color.lstrip("#")
    r, g, b = (int(color[i:i + 2], 16) for i in (0, 2, 4))
    return f"rgba({r},{g},{b},{alpha})"


def _metric_exists(df: pd.DataFrame, metric: str) -> bool:
    return metric in df.columns and pd.to_numeric(df[metric], errors="coerce").notna().any()


def plot_mean_band(
    df: pd.DataFrame,
    metric: str,
    *,
    x_col: str = "epoch",
    group_col: str = "entropy_beta",
    run_col: str = "run_name",
    title: str | None = None,
    yaxis_title: str | None = None,
    band: str = "std",
    yaxis_type: str = "linear",
    show_individual: bool = True,
):
    if not _metric_exists(df, metric):
        print(f"skip {metric}: column is missing or all values are NaN")
        return None
    required = [x_col, group_col, run_col, metric]
    missing = [col for col in required if col not in df.columns]
    if missing:
        print(f"skip {metric}: missing columns {missing}")
        return None

    view = df[required].copy()
    view[metric] = pd.to_numeric(view[metric], errors="coerce")
    view = view.dropna(subset=[x_col, group_col, metric])
    if view.empty:
        print(f"skip {metric}: no plottable rows")
        return None

    figure = go.Figure()
    for color_index, (group_value, group_df) in enumerate(view.groupby(group_col, sort=True)):
        color = COLORS[color_index % len(COLORS)]
        label = f"beta={group_value:g}" if isinstance(group_value, (int, float, np.floating)) else str(group_value)

        if show_individual:
            for _, run_df in group_df.groupby(run_col, sort=False):
                run_df = run_df.sort_values(x_col)
                figure.add_trace(
                    go.Scatter(
                        x=run_df[x_col],
                        y=run_df[metric],
                        mode="lines",
                        line={"color": color, "width": 1},
                        opacity=0.22,
                        name=label,
                        legendgroup=label,
                        showlegend=False,
                        hovertemplate=(
                            f"{label}<br>{run_col}=%{{customdata}}<br>"
                            f"{x_col}=%{{x}}<br>{metric}=%{{y:.6g}}<extra></extra>"
                        ),
                        customdata=run_df[run_col],
                    )
                )

        stats = (
            group_df.groupby(x_col, as_index=False)[metric]
            .agg(["mean", "std", "count"])
            .reset_index()
            .sort_values(x_col)
        )
        spread = stats["std"].fillna(0.0)
        if band == "sem":
            spread = spread / np.sqrt(stats["count"].clip(lower=1))
        upper = stats["mean"] + spread
        lower = stats["mean"] - spread

        figure.add_trace(
            go.Scatter(
                x=stats[x_col],
                y=upper,
                mode="lines",
                line={"width": 0},
                legendgroup=label,
                showlegend=False,
                hoverinfo="skip",
            )
        )
        figure.add_trace(
            go.Scatter(
                x=stats[x_col],
                y=lower,
                mode="lines",
                fill="tonexty",
                fillcolor=_hex_to_rgba(color),
                line={"width": 0},
                legendgroup=label,
                showlegend=False,
                hoverinfo="skip",
            )
        )
        figure.add_trace(
            go.Scatter(
                x=stats[x_col],
                y=stats["mean"],
                mode="lines",
                line={"color": color, "width": 3},
                name=f"{label} mean",
                legendgroup=label,
                hovertemplate=f"{label}<br>{x_col}=%{{x}}<br>mean {metric}=%{{y:.6g}}<extra></extra>",
            )
        )

    figure.update_layout(
        title=title or f"{metric} by {group_col}",
        xaxis_title=x_col,
        yaxis_title=yaxis_title or metric,
        yaxis_type=yaxis_type,
        hovermode="x unified",
        legend={"x": 1.02, "y": 1.0, "xanchor": "left", "yanchor": "top"},
        margin={"r": 260},
        template="plotly_white",
    )
    figure.show()
    return figure


def plot_beta_box(df: pd.DataFrame, metric: str, *, title: str | None = None):
    if not _metric_exists(df, metric):
        print(f"skip {metric}: column is missing or all values are NaN")
        return None
    figure = go.Figure()
    for color_index, (beta, group_df) in enumerate(df.groupby("entropy_beta", sort=True)):
        color = COLORS[color_index % len(COLORS)]
        label = f"beta={beta:g}" if isinstance(beta, (int, float, np.floating)) else str(beta)
        figure.add_trace(
            go.Box(
                y=pd.to_numeric(group_df[metric], errors="coerce"),
                name=label,
                boxpoints="all",
                jitter=0.25,
                pointpos=0.0,
                marker={"color": color, "size": 8},
                line={"color": color},
                customdata=group_df[["run_name", "repeat_idx"]],
                hovertemplate=(
                    f"{label}<br>{metric}=%{{y:.6g}}<br>"
                    "run=%{customdata[0]}<br>repeat=%{customdata[1]}<extra></extra>"
                ),
            )
        )
    figure.update_layout(
        title=title or metric,
        yaxis_title=metric,
        template="plotly_white",
    )
    figure.show()
    return figure


def plot_tradeoff(df: pd.DataFrame, x_metric: str, y_metric: str, *, title: str | None = None):
    missing = [metric for metric in (x_metric, y_metric) if not _metric_exists(df, metric)]
    if missing:
        print(f"skip tradeoff: missing or empty metrics {missing}")
        return None
    figure = go.Figure()
    for color_index, (beta, group_df) in enumerate(df.groupby("entropy_beta", sort=True)):
        color = COLORS[color_index % len(COLORS)]
        label = f"beta={beta:g}" if isinstance(beta, (int, float, np.floating)) else str(beta)
        figure.add_trace(
            go.Scatter(
                x=pd.to_numeric(group_df[x_metric], errors="coerce"),
                y=pd.to_numeric(group_df[y_metric], errors="coerce"),
                mode="markers+text",
                text=group_df["repeat_idx"].astype(str),
                textposition="top center",
                marker={"color": color, "size": 11},
                name=label,
                customdata=group_df[["run_name", "best_epoch", "final_epoch"]],
                hovertemplate=(
                    f"{label}<br>{x_metric}=%{{x:.6g}}<br>{y_metric}=%{{y:.6g}}<br>"
                    "run=%{customdata[0]}<br>best_epoch=%{customdata[1]}<br>final_epoch=%{customdata[2]}<extra></extra>"
                ),
            )
        )
    figure.update_layout(
        title=title or f"{y_metric} vs {x_metric}",
        xaxis_title=x_metric,
        yaxis_title=y_metric,
        template="plotly_white",
    )
    figure.show()
    return figure

In [ ]:
TRAJECTORY_PLOTS = [
    (ACC_COL, "Validation accuracy by beta", "linear"),
    ("valid_loss", "Validation loss by beta", "linear"),
    ("valid_aig_flops_active_ratio", "Active FLOPs ratio by beta", "linear"),
    ("valid_aig_flops_skip_ratio", "Skipped FLOPs ratio by beta", "linear"),
    ("valid_active_blocks_expected", "Expected active AIG blocks by beta", "linear"),
    ("valid_mean_gate_prob", "Mean gate probability by beta", "linear"),
    ("valid_entropy", "Posterior entropy by beta", "linear"),
    ("valid_negative_entropy", "Negative posterior entropy by beta", "linear"),
    ("valid_mean_p_open", "Mean posterior p(open) by beta", "linear"),
    ("valid_lambda_p_open_loss", "lambda * mean_p_open term by beta", "log"),
    ("valid_entropy_loss_term", "beta * negative_entropy term by beta", "linear"),
    ("valid_estimated_gate_loss", "Estimated gate loss term by beta", "linear"),
]

for metric, title, yaxis_type in TRAJECTORY_PLOTS:
    plot_mean_band(
        history_df,
        metric,
        title=title,
        yaxis_type=yaxis_type,
        band="std",
        show_individual=True,
    )

In [ ]:
SUMMARY_METRICS = [
    "best_valid_accuracy",
    "final_valid_accuracy",
    "final_valid_loss",
    "final_valid_entropy",
    "final_valid_mean_p_open",
    "final_valid_entropy_loss_term",
    "final_valid_estimated_gate_loss",
    "final_valid_active_blocks_expected",
    "final_valid_mean_gate_prob",
    "final_valid_aig_flops_active_ratio",
    "final_valid_aig_flops_skip_ratio",
    "final_lambda_coef",
]
SUMMARY_METRICS = [metric for metric in SUMMARY_METRICS if metric in run_summary_df.columns]

beta_summary = (
    run_summary_df.groupby("entropy_beta")[SUMMARY_METRICS]
    .agg(["mean", "std", "min", "max"])
    .sort_index()
)
beta_summary.columns = ["_".join(col).strip("_") for col in beta_summary.columns]
display(beta_summary)

if 0.0 in beta_summary.index:
    baseline = beta_summary.loc[0.0]
    effect_rows = []
    for beta, row in beta_summary.iterrows():
        effect = {"entropy_beta": beta}
        for metric in SUMMARY_METRICS:
            mean_col = f"{metric}_mean"
            if mean_col in beta_summary.columns:
                effect[f"delta_{metric}_mean_vs_beta0"] = row[mean_col] - baseline[mean_col]
        effect_rows.append(effect)
    effect_df = pd.DataFrame(effect_rows).set_index("entropy_beta")
    display(effect_df)
else:
    print("No beta=0 baseline found for delta table.")

In [ ]:
BOX_METRICS = [
    "best_valid_accuracy",
    "final_valid_accuracy",
    "final_valid_entropy",
    "final_valid_mean_p_open",
    "final_valid_aig_flops_active_ratio",
    "final_valid_aig_flops_skip_ratio",
    "final_valid_active_blocks_expected",
    "final_valid_estimated_gate_loss",
]
for metric in BOX_METRICS:
    plot_beta_box(run_summary_df, metric)

In [ ]:
COMPUTE_METRIC = _find_col(
    run_summary_df,
    (
        "final_valid_aig_flops_active_ratio",
        "final_valid_active_blocks_expected",
        "final_valid_mean_gate_prob",
        "final_valid_mean_p_open",
    ),
)

if COMPUTE_METRIC is None:
    print("No compute/gate endpoint metric found for tradeoff plots.")
else:
    plot_tradeoff(
        run_summary_df,
        COMPUTE_METRIC,
        "best_valid_accuracy",
        title=f"Best accuracy vs {COMPUTE_METRIC}",
    )
    plot_tradeoff(
        run_summary_df,
        COMPUTE_METRIC,
        "final_valid_accuracy",
        title=f"Final accuracy vs {COMPUTE_METRIC}",
    )

if "final_valid_entropy" in run_summary_df.columns:
    plot_tradeoff(
        run_summary_df,
        "final_valid_entropy",
        "best_valid_accuracy",
        title="Best accuracy vs final posterior entropy",
    )

In [ ]:
def add_gradient_ratio(
    df: pd.DataFrame,
    output_col: str,
    numerator_col: str,
    denominator_col: str,
) -> pd.DataFrame:
    df = df.copy()
    if numerator_col not in df.columns or denominator_col not in df.columns:
        print(f"skip {output_col}: missing {numerator_col} or {denominator_col}")
        return df
    numerator = pd.to_numeric(df[numerator_col], errors="coerce")
    denominator = pd.to_numeric(df[denominator_col], errors="coerce").replace(0.0, np.nan)
    df[output_col] = numerator / denominator
    return df


history_df = add_gradient_ratio(
    history_df,
    "grad_ratio_regularization_to_ce_total_mean",
    "grad_norm_regularization_total_mean",
    "grad_norm_ce_total_mean",
)
history_df = add_gradient_ratio(
    history_df,
    "grad_ratio_regularization_to_ce_gumbel_logits_mean",
    "grad_norm_regularization_gumbel_logits_total_mean",
    "grad_norm_ce_gumbel_logits_total_mean",
)
history_df = add_gradient_ratio(
    history_df,
    "grad_ratio_total_to_ce_total_mean",
    "grad_norm_total_total_mean",
    "grad_norm_ce_total_mean",
)

for metric in (
    "grad_ratio_regularization_to_ce_total_mean",
    "grad_ratio_regularization_to_ce_gumbel_logits_mean",
    "grad_ratio_total_to_ce_total_mean",
):
    plot_mean_band(
        history_df,
        metric,
        title=metric,
        yaxis_type="log",
        band="std",
        show_individual=True,
    )

## What to look for

Use beta `0.0` as the no-entropy baseline. Evidence that entropy contributed is stronger if several views agree:

- `valid_entropy` or `valid_negative_entropy` shifts consistently by beta;
- `valid_mean_p_open`, active blocks, or active FLOPs move coherently rather than randomly across repeats;
- best/final accuracy changes are larger than the repeat spread at beta `0.0`;
- accuracy-vs-compute scatter moves to a better frontier;
- `grad(regularization) / grad(CE)` changes around gate parameters, showing that the entropy term is actually large enough to affect optimization.